In [1]:
!ls
!python generate_data.py

generate_data.py  mock_llm.py  sample_data
generated: customers=1200 products=40 orders_raw=13084 tickets=164


In [2]:
import pandas as pd, numpy as np

cust = pd.read_csv("data/raw/customers.csv")
prod = pd.read_csv("data/raw/products.csv")
orders = pd.read_csv("data/raw/orders.csv")

for name, df in [("customer", cust),("products",prod),("orders", orders)]:
  print(name, df.shape)
  print(pd.DataFrame({"dtype": df.dtypes.astype(str),"missing": df.isna().sum(), "missing_%": (df.isna().mean()*100).round(2)}))


customer (1200, 5)
               dtype  missing  missing_%
customer_id   object        0       0.00
city          object        0       0.00
signup_date   object        0       0.00
age          float64       61       5.08
gender        object       30       2.50
products (40, 4)
               dtype  missing  missing_%
product_id    object        0        0.0
product_name  object        0        0.0
category      object        0        0.0
price_inr      int64        0        0.0
orders (13084, 9)
                    dtype  missing  missing_%
order_id           object        0       0.00
customer_id        object        0       0.00
product_id         object        0       0.00
quantity            int64        0       0.00
order_ts           object        0       0.00
status             object        0       0.00
discount_pct        int64        0       0.00
order_amount       object       53       0.41
delivery_minutes  float64     2541      19.42


**Auditing**

In [3]:
audit = {
    "raw orders rows": len(orders),
    "exact duplicate rows": orders.duplicated().sum(),
    "pending rows": (orders.status == "pending").sum(),
    "distinct order_id": orders.order_id.nunique(),
    "non-numeric order_amount": pd.to_numeric(orders.order_amount, errors="coerce").isna().sum(),
    "negative delivery_minutes": (orders.delivery_minutes < 0).sum(),
    "blank delivery_minutes": orders.delivery_minutes.isna().sum(),
    "blank age": cust.age.isna().sum(),
    "age outside 10-100": ((cust.age < 10) | (cust.age > 100)).sum(),
    "raw city spellings": cust.city.nunique(),
    "raw gender spellings": cust.gender.dropna().nunique(),
    "blank gender": cust.gender.isna().sum(),
}
for k, v in audit.items():
    print(f"{k:30s} {v}")


raw orders rows                13084
exact duplicate rows           119
pending rows                   656
distinct order_id              12309
non-numeric order_amount       120
negative delivery_minutes      106
blank delivery_minutes         2541
blank age                      61
age outside 10-100             6
raw city spellings             24
raw gender spellings           10
blank gender                   30


**Cleaning the Customers Data**

In [6]:
c = cust.copy()


In [7]:
print(c["city"].value_counts())
print(c["gender"].value_counts())
print(c["age"].isna().sum(), c["age"].min(), c["age"].max())


city
Pune           148
Delhi          123
Hyderabad      119
Bengaluru      115
Mumbai         113
Chennai        106
chennai         44
hyderabad       40
delhi           36
mumbai          35
pune            35
BENGALURU       32
 Delhi          30
bengaluru       27
PUNE            24
 Bengaluru      23
DELHI           21
CHENNAI         21
 Chennai        20
 Pune           20
 Mumbai         19
 Hyderabad      18
MUMBAI          17
HYDERABAD       14
Name: count, dtype: int64
gender
female    136
FEMALE    122
F         120
m         118
f         117
Female    115
male      114
MALE      110
M         110
Male      108
Name: count, dtype: int64
61 -4.0 999.0


In [9]:
c = cust.copy()

c["city"] = c["city"].str.strip().str.title()

g = c["gender"].fillna("").str.strip().str.lower()
c["gender"] = g.map({"male": "male", "m": "male", "female": "female", "f": "female"}).fillna("unknown")

c["age"] = c["age"].where(c["age"].between(10, 100))
c["age"] = c["age"].fillna(c.groupby("city")["age"].transform("median"))

c["signup_date"] = pd.to_datetime(c["signup_date"])


In [12]:
print(c["city"].value_counts())

print(c["gender"].value_counts())

print(c["age"].isna().sum(), c["age"].min(), c["age"].max())


city
Pune         227
Delhi        210
Bengaluru    197
Chennai      191
Hyderabad    191
Mumbai       184
Name: count, dtype: int64
gender
female     610
male       560
unknown     30
Name: count, dtype: int64
0 18.0 60.0


**Cleaning the Orders**

In [13]:
o = orders.copy()

In [14]:
o["order_ts"] = pd.to_datetime(o["order_ts"])

In [21]:
print(o["order_ts"].dtype)
print(o["order_ts"].head(3))
print(o["order_ts"].min(), "->", o["order_ts"].max())
print("unparseable timestamps:", o["order_ts"].isna().sum())


datetime64[ns]
0   2025-09-16 08:07:21
1   2025-09-03 07:34:01
2   2025-04-08 17:08:14
Name: order_ts, dtype: datetime64[ns]
2025-04-01 07:08:28 -> 2025-09-30 22:46:26
unparseable timestamps: 0


In [15]:
o = o.drop_duplicates()

In [16]:
o = o.sort_values("order_ts").drop_duplicates("order_id", keep="last")

In [20]:
o = orders.copy()
log = {}

o["order_ts"] = pd.to_datetime(o["order_ts"])

n = len(o)
o = o.drop_duplicates()
log["ii. exact duplicates removed"] = n - len(o)

n = len(o)
o = o.sort_values("order_ts").drop_duplicates("order_id", keep="last")
log["iii. earlier records of the same order removed"] = n - len(o)

o["order_amount"] = pd.to_numeric(o["order_amount"], errors="coerce")
o = o.merge(prod[["product_id", "category", "price_inr"]], on="product_id", how="left")
miss = o["order_amount"].isna()
calc = (o["price_inr"] * o["quantity"] * (1 - o["discount_pct"] / 100)).round(2)
o.loc[miss, "order_amount"] = calc[miss]
log["iv. missing amounts filled from price"] = int(miss.sum())

neg = o["delivery_minutes"] < 0
o.loc[neg, "delivery_minutes"] = np.nan
log["v. negative delivery_minutes set to missing"] = int(neg.sum())

comp = o["status"] == "completed"
med = o[comp].groupby("category")["delivery_minutes"].transform("median")
need = comp & o["delivery_minutes"].isna()
o.loc[comp, "delivery_minutes"] = o.loc[comp, "delivery_minutes"].fillna(med)
log["v. completed delivery_minutes filled (category median)"] = int(need.sum())

o = o.drop(columns=["category", "price_inr"]).sort_values("order_id").reset_index(drop=True)

for k, v in log.items():
    print(f"{k:55s} {v}")

# checkpoint
done = o[o["status"] == "completed"]
canc = o[o["status"] == "cancelled"]
print("\nrows:", len(o), o["status"].value_counts().to_dict())
print("missing amount:", o["order_amount"].isna().sum())
print("missing delivery (completed, cancelled):", done["delivery_minutes"].isna().sum(), canc["delivery_minutes"].isna().sum())
print("revenue INR:", round(done["order_amount"].sum(), 2))

ii. exact duplicates removed                            119
iii. earlier records of the same order removed          656
iv. missing amounts filled from price                   112
v. negative delivery_minutes set to missing             105
v. completed delivery_minutes filled (category median)  774

rows: 12309 {'completed': 11110, 'cancelled': 1199}
missing amount: 0
missing delivery (completed, cancelled): 0 1199
revenue INR: 2998668.8


In [22]:
import os, sqlite3
os.makedirs("data/clean", exist_ok=True)

o_out = o.copy()
o_out["order_ts"] = o_out["order_ts"].dt.strftime("%Y-%m-%d %H:%M:%S")
c_out = c.copy()
c_out["signup_date"] = c_out["signup_date"].dt.strftime("%Y-%m-%d")

c_out.to_csv("data/clean/customers.csv", index=False)
prod.to_csv("data/clean/products.csv", index=False)
o_out.to_csv("data/clean/orders.csv", index=False)

# SQLite database
if os.path.exists("data/instamart.db"):
    os.remove("data/instamart.db")
con = sqlite3.connect("data/instamart.db")
c_out.to_sql("customers", con, index=False)
prod.to_sql("products", con, index=False)
o_out.to_sql("orders", con, index=False)
con.close()
print("saved")


saved


In [23]:
c2 = pd.read_csv("data/clean/customers.csv")
o2 = pd.read_csv("data/clean/orders.csv")
con = sqlite3.connect("data/instamart.db")

for t, n in [("customers", 1200), ("products", 40), ("orders", 12309)]:
    assert pd.read_sql_query(f"SELECT COUNT(*) n FROM {t}", con)["n"][0] == n

assert len(o2) == 12309 and o2["order_id"].is_unique
vc = o2["status"].value_counts()
assert vc["completed"] == 11110 and vc["cancelled"] == 1199 and "pending" not in vc.index
assert o2["order_amount"].isna().sum() == 0
d2, k2 = o2[o2.status == "completed"], o2[o2.status == "cancelled"]
assert d2["delivery_minutes"].isna().sum() == 0 and k2["delivery_minutes"].isna().sum() == 1199
assert abs(d2["order_amount"].sum() - 2998668.80) < 0.01
assert c2["city"].value_counts().to_dict() == {"Pune": 227, "Delhi": 210, "Bengaluru": 197,
                                               "Chennai": 191, "Hyderabad": 191, "Mumbai": 184}
assert c2["gender"].value_counts().to_dict() == {"female": 610, "male": 560, "unknown": 30}
assert c2["age"].isna().sum() == 0 and c2["age"].between(18, 60).all()

print("ALL CHECKPOINTS PASSED")


ALL CHECKPOINTS PASSED


In [26]:
!pwd
!ls
!find / -name "queries.sql" 2>/dev/null

/content
data  generate_data.py	mock_llm.py  sample_data
/sql/queries.sql


In [27]:
import shutil, os
os.makedirs("/content/sql", exist_ok=True)
shutil.move("/sql/queries.sql", "/content/sql/queries.sql")
!ls /content/sql

queries.sql


In [29]:
import re, sqlite3, pandas as pd

con = sqlite3.connect("data/instamart.db")
text = open("sql/queries.sql", encoding="utf-8").read()
parts = re.split(r"^-- (Q\d)[^\n]*\n", text, flags=re.M)
queries = dict(zip(parts[1::2], parts[2::2]))
res = {q: pd.read_sql_query(sql, con) for q, sql in queries.items()}

print("Q1 customers:", len(res["Q1"]), "(ref 30)")

q2 = res["Q2"]
print("Q2 rows:", len(q2), "| total:", round(q2.revenue.sum(), 2), "| largest:", q2.iloc[0].to_dict())

print("Q3 products:", len(res["Q3"]), "(ref 11)")

print("Q4 customers:", len(res["Q4"]), "(ref 25)")

q5 = res["Q5"]
print("Q5 customers:", len(q5), "| average:", q5.avg_total.iloc[0], "(ref 550, 2552.06)")

q6 = res["Q6"]
print(q6[q6.row_num == 1][["city", "customer_id", "total_spend"]].to_string(index=False))

print("Q6 rows where RANK != ROW_NUMBER:", int((q6.rnk != q6.row_num).sum()), "(ref 8)")

q7 = res["Q7"]
print("Q7 rows:", len(q7), "| last cumulative:", q7.cumulative_revenue.iloc[-1])

Q1 customers: 30 (ref 30)
Q2 rows: 30 | total: 2998668.8 | largest: {'city': 'Bengaluru', 'category': 'Electronics', 'revenue': 233909.45}
Q3 products: 11 (ref 11)
Q4 customers: 25 (ref 25)
Q5 customers: 550 | average: 2552.06 (ref 550, 2552.06)
     city customer_id  total_spend
Bengaluru       C0082     12856.20
  Chennai       C0930     10151.40
    Delhi       C0291     13742.60
Hyderabad       C0743     12730.05
   Mumbai       C0693     14766.55
     Pune       C0426     10377.10
Q6 rows where RANK != ROW_NUMBER: 8 (ref 8)
Q7 rows: 183 | last cumulative: 2998668.8


In [30]:
print(q2.iloc[0].to_dict())

{'city': 'Bengaluru', 'category': 'Electronics', 'revenue': 233909.45}


In [32]:
c2 = pd.read_csv("data/clean/customers.csv")
p2 = pd.read_csv("data/clean/products.csv")
o2 = pd.read_csv("data/clean/orders.csv")
done = o2[o2.status == "completed"]
m = done.merge(c2[["customer_id", "city"]], on="customer_id").merge(p2[["product_id", "category"]], on="product_id")

# (a) pivot table and SQL total
pv = pd.pivot_table(m, values="order_amount", index="city", columns="category", aggfunc="sum", margins=True)
print(pv.round(2))
print("pivot grand total:", round(pv.loc["All", "All"], 2), "| SQL total:", round(q2.revenue.sum(), 2))

# per-city sum, mean, count
city_stats = m.groupby("city").agg(total=("order_amount", "sum"),
                                   mean=("order_amount", "mean"),
                                   count=("order_amount", "count")).round(2)
print(city_stats)

# (b) distribution statistics
a = done["order_amount"]
q1, q3 = a.quantile(0.25), a.quantile(0.75)
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"mean {a.mean():.2f} | median {a.median():.2f} | skew {a.skew():.2f}")
print(f"Q1 {q1:.2f} | Q3 {q3:.2f} | fences [{lo:.2f}, {hi:.2f}]")
print("outliers below:", (a < lo).sum(), "| above:", (a > hi).sum())

# correlations
print(done[["order_amount", "discount_pct", "quantity", "delivery_minutes"]].corr().round(3))


category       Dairy  Electronics    Grocery  Personal Care     Snacks  \
city                                                                     
Bengaluru   87075.05    233909.45  110915.65       20193.85   71549.40   
Chennai     78301.80    195508.80   93817.45       27675.40   68325.70   
Delhi      117271.70    158549.10  112658.05       23471.50   90879.45   
Hyderabad   83731.00    228913.10  109475.40       24800.05   81670.35   
Mumbai      75209.85    214738.40   78970.15       38018.00   72122.10   
Pune        90349.55    187629.25  108456.60       34275.15   80207.50   
All        531938.95   1219248.10  614293.30      168433.95  464754.50   

category          All  
city                   
Bengaluru   523643.40  
Chennai     463629.15  
Delhi       502829.80  
Hyderabad   528589.90  
Mumbai      479058.50  
Pune        500918.05  
All        2998668.80  
pivot grand total: 2998668.8 | SQL total: 2998668.8
               total    mean  count
city                         

In [33]:
big = m[m["order_amount"] > 645.30]
print(big["category"].value_counts())


category
Electronics      635
Dairy            166
Grocery           52
Personal Care     44
Name: count, dtype: int64


**What does this data relate to?**
This data relates to 11,110 Instamart order transactions from six cities in the months of April through September 2025. The order data has the amount, the discount, the quantity, and the delivery time.

**What are the distributions and statistics?**
The data has most of the orders as small. The mean (INR 269.91) is twice the median (INR 135.00), and the skewness (4.61) indicates that the distribution is positively skewed.

**Are there any outliers?** There are 897 outliers above the upper fence of INR 645.30, and none below the lower fence. Around 71% of them (635) are Electronics, so they are statistical outliers but genuine high-value orders, not data errors.

**How are the variables related to each other?** There are no strong relationships between the variables. The strongest is between amount and discount (-0.095), which is very weak: higher-value orders tend to have slightly lower discounts. A correlation this small does not establish causation.